# Stage 04 â€” DoubleML Extension

Estimate causal effects using Double/Debiased Machine Learning (DML).
Follows Baiardi & Naghi (2024, Econometrics Journal) methodology:
- 7 ML methods: Lasso, DecisionTree, Boosting, Forest, NeuralNet, Ensemble, Best
- Median aggregation across repeated sample splits with B&N adjusted SE
- BLP heterogeneity test, 5-quintile GATE, CLAN
- Causal forest (skipped gracefully if econml not installed)

Reference: B&N Table 2 (N=63, 17 controls, K=2 folds, 100 reps)

In [1]:
# â”€â”€ Cell 1: Imports and data loading â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from paths import *
import json, yaml, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import doubleml as dml
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.linear_model import LassoCV
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy import stats
import statsmodels.api as sm

# Causal forest (skip gracefully if not installed)
try:
    from econml.dml import CausalForestDML
    ECONML_AVAILABLE = True
except ImportError:
    ECONML_AVAILABLE = False
    print("econml not installed â€” causal forest will be skipped")

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Load data and specs
df = pd.read_parquet(str(DATASET_PATH))
spec = json.loads(PAPER_SPEC.read_text())
config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text())
rep = json.loads(REPLICATION_RESULTS.read_text())

# Extract identification info
outcome = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
controls = spec['identification']['controls']
id_type = spec['identification']['type']
model_type = spec['dml']['model_type']

# Additional treatments from spec
additional_treatments = spec['identification'].get('additional_treatments', [])

# Build specification grid: primary + additional treatments
treatments_list = [{"variable": treatment, "label": spec['identification'].get('treatment_label', treatment)}]
for extra in additional_treatments:
    treatments_list.append(extra)

specs_grid = []
for t in treatments_list:
    specs_grid.append({
        "outcome": {"variable": outcome, "label": spec['identification'].get('outcome_label', outcome)},
        "treatment": t,
        "controls": controls
    })

print(f"Model type: {model_type} (DoubleMLPLR)")
print(f"Specification grid: {len(specs_grid)} specs")
for i, s in enumerate(specs_grid):
    print(f"  Spec {i+1}: {s['outcome']['variable']} ~ {s['treatment']['variable']}")
print(f"Controls ({len(controls)}): {controls}")

# Prepare clean dataset
key_cols = [outcome] + [t['variable'] for t in treatments_list] + controls
df_clean = df[key_cols].dropna().reset_index(drop=True)
n_obs = len(df_clean)
print(f"\nClean dataset: N={n_obs}, K={len(controls)} controls, zero missing")

econml not installed â€” causal forest will be skipped


Model type: PLR (DoubleMLPLR)
Specification grid: 3 specs
  Spec 1: growth ~ skill1_corr
  Spec 2: growth ~ diffa
  Spec 3: growth ~ diffg
Controls (17): ['avg_tar', 'init', 'inv', 'human_cap', 'w_africa', 'e_africa', 's_c_africa', 'n_afr_me', 'e_europe', 'lat_america', 'e_asia', 's_e_asia', 's_w_asia', 'dum80_83', 'dum85_87', 'ln_init_q_skilla', 'ln_init_q_unskilla']

Clean dataset: N=63, K=17 controls, zero missing


In [2]:
# ── Cell 2: DML configuration and learner builder ────────────────────────────

# Cross-fitting parameters from config (B&N: K=2 for N<200)
n_folds = config['dml'].get('n_folds', 2)
if n_folds == "auto" or n_folds is None:
    n_folds = 2 if n_obs < 200 else 5
n_rep = config['dml'].get('n_rep', 20)

## Revision Round 1
# REMOVED: n_rep = 5 override.  Use config value (n_rep=20) as recommended
# by Referee 2.  With n_rep=5 and K=2 (N=63), the median coefficient has
# only 4 effective degrees of freedom—insufficient for stable inference in
# specifications with p-values near conventional thresholds.
print(f"Cross-fitting: K={n_folds} folds, n_rep={n_rep}")
print(f"  (B&N use K=2, 100 reps; we use {n_rep} per config)")

def build_learners(n_obs, n_features):
    """Build dictionary of 5 base ML learners following B&N methodology."""
    learners = {
        "Lasso": LassoCV(cv=min(5, max(2, n_obs // 10)), max_iter=10000),
        "DecisionTree": DecisionTreeRegressor(
            ccp_alpha=0.01, max_depth=min(5, max(2, n_obs // 10)),
            random_state=42
        ),
        "Boosting": GradientBoostingRegressor(
            n_estimators=500, learning_rate=0.01,
            max_depth=2, subsample=0.5,
            min_samples_leaf=max(1, n_obs // 20),
            random_state=42
        ),
        "Forest": RandomForestRegressor(
            n_estimators=500, min_samples_leaf=5,
            max_features="sqrt", n_jobs=-1,
            random_state=42
        ),
        "NeuralNet": MLPRegressor(
            hidden_layer_sizes=(64,), max_iter=2000,
            early_stopping=True, learning_rate_init=0.01,
            random_state=42, validation_fraction=0.2
        ),
    }
    return learners

print(f"Learners: {list(build_learners(n_obs, len(controls)).keys())} + Ensemble + Best")

Cross-fitting: K=2 folds, n_rep=20
  (B&N use K=2, 100 reps; we use 20 per config)
Learners: ['Lasso', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet'] + Ensemble + Best


In [3]:
# ── Cell 3: Run DML for all specs x all learners ─────────────────────────────

all_results = []

for spec_idx, s in enumerate(specs_grid):
    y_col = s["outcome"]["variable"]
    d_col = s["treatment"]["variable"]
    
    print(f"\n{'='*70}")
    print(f"Spec {spec_idx+1}/{len(specs_grid)}: {y_col} ~ {d_col}")
    print(f"{'='*70}")
    
    # Listwise deletion for this specification
    cols_needed = [y_col, d_col] + controls
    df_spec = df_clean[cols_needed].dropna().reset_index(drop=True)
    n_spec = len(df_spec)
    
    # Adaptive K for this spec
    k_folds = 2 if n_spec < 200 else n_folds
    
    # Build DoubleML data object
    dml_data = dml.DoubleMLData(
        df_spec, y_col=y_col, d_cols=d_col, x_cols=controls
    )
    
    learner_results = {}
    learner_nuisance_mse = {}
    learner_objects = {}  # store fitted objects for HTE
    
    for learner_name, ml_model in build_learners(n_spec, len(controls)).items():
        print(f"\n  Fitting {learner_name}...", end=" ")
        
        # Clone for ml_m (each nuisance equation needs its own learner)
        from sklearn.base import clone
        ml_l = clone(ml_model)
        ml_m = clone(ml_model)
        
        # PLR for OLS identification
        obj = dml.DoubleMLPLR(
            dml_data, ml_l=ml_l, ml_m=ml_m,
            n_folds=k_folds, n_rep=n_rep
        )
        
        obj.fit(store_models=(learner_name == "Lasso"))
        
        # Extract per-rep estimates for median aggregation
        all_coefs = obj.all_coef.flatten()  # shape (n_rep,)
        all_ses = obj.all_se.flatten()      # shape (n_rep,)
        
        median_coef = float(np.median(all_coefs))
        # B&N adjusted SE: median(sqrt(SE_k^2 + (coef_k - median_coef)^2))
        se_adj = float(np.median(np.sqrt(all_ses**2 + (all_coefs - median_coef)**2)))
        
        # Nuisance model quality (use cross-fitted predictions, averaged over reps)
        preds_outcome = obj.predictions['ml_l'][:, :, 0].mean(axis=1)   # avg over reps
        preds_treatment = obj.predictions['ml_m'][:, :, 0].mean(axis=1)
        
        r2_y = float(r2_score(df_spec[y_col], preds_outcome))
        r2_d = float(r2_score(df_spec[d_col], preds_treatment))
        mse_y = float(mean_squared_error(df_spec[y_col], preds_outcome))
        mse_d = float(mean_squared_error(df_spec[d_col], preds_treatment))
        
        learner_nuisance_mse[learner_name] = {
            "mse_outcome": mse_y, "mse_treatment": mse_d,
            "r2_outcome": r2_y, "r2_treatment": r2_d,
        }
        
        # Lasso coefficient diagnostics
        lasso_diag = None
        if learner_name == "Lasso":
            try:
                lasso_model_l = obj.models['ml_l'][d_col][0][0]
                coefs_l = pd.Series(lasso_model_l.coef_, index=controls)
                nonzero_l = coefs_l[coefs_l != 0].sort_values(key=abs, ascending=False)
                
                lasso_model_m = obj.models['ml_m'][d_col][0][0]
                coefs_m = pd.Series(lasso_model_m.coef_, index=controls)
                nonzero_m = coefs_m[coefs_m != 0].sort_values(key=abs, ascending=False)
                
                lasso_diag = {
                    "outcome_model": {
                        "n_nonzero": int(len(nonzero_l)),
                        "total_p": len(controls),
                        "top_variables": {k: round(v, 6) for k, v in nonzero_l.head(5).items()},
                    },
                    "treatment_model": {
                        "n_nonzero": int(len(nonzero_m)),
                        "total_p": len(controls),
                        "top_variables": {k: round(v, 6) for k, v in nonzero_m.head(5).items()},
                    },
                }
                print(f"Lasso selected {len(nonzero_l)}/{len(controls)} vars (outcome), "
                      f"{len(nonzero_m)}/{len(controls)} vars (treatment)")
            except Exception as e:
                print(f"  (Lasso diagnostics failed: {e})")
        
        pval = float(2 * (1 - stats.norm.cdf(abs(median_coef / se_adj)))) if se_adj > 0 else 1.0
        
        learner_results[learner_name] = {
            "coef": median_coef,
            "se": se_adj,
            "ci_lo": median_coef - 1.96 * se_adj,
            "ci_hi": median_coef + 1.96 * se_adj,
            "pval": pval,
            "per_rep_coefs": all_coefs.tolist(),
            "per_rep_ses": all_ses.tolist(),
            "nuisance": learner_nuisance_mse[learner_name],
            "lasso_diagnostics": lasso_diag,
        }
        
        # Store the last-fitted object for HTE (primary spec only)
        if spec_idx == 0:
            learner_objects[learner_name] = {
                "obj": obj, "preds_outcome": preds_outcome,
                "preds_treatment": preds_treatment, "df_spec": df_spec,
            }
        
        stars = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.10 else ""))
        print(f"coef={median_coef:.4f}, se={se_adj:.4f}, p={pval:.4f} {stars}")
        print(f"    Nuisance R²: outcome={r2_y:.3f}, treatment={r2_d:.3f}")
    
    # ── Ensemble: MSE-inverse-weighted combination ──
    methods_for_ensemble = [k for k in learner_results if k not in ("Ensemble", "Best")]
    weights_outcome = {}
    for m in methods_for_ensemble:
        mse = learner_nuisance_mse[m]["mse_outcome"]
        weights_outcome[m] = 1.0 / max(mse, 1e-10)
    total_w = sum(weights_outcome.values())
    weights_outcome = {k: v / total_w for k, v in weights_outcome.items()}
    
    ensemble_coef = sum(weights_outcome[m] * learner_results[m]["coef"] for m in methods_for_ensemble)
    ensemble_se = sum(weights_outcome[m] * learner_results[m]["se"] for m in methods_for_ensemble)
    ensemble_pval = float(2 * (1 - stats.norm.cdf(abs(ensemble_coef / ensemble_se)))) if ensemble_se > 0 else 1.0
    
    learner_results["Ensemble"] = {
        "coef": float(ensemble_coef),
        "se": float(ensemble_se),
        "ci_lo": float(ensemble_coef - 1.96 * ensemble_se),
        "ci_hi": float(ensemble_coef + 1.96 * ensemble_se),
        "pval": ensemble_pval,
        "weights": {k: round(v, 3) for k, v in weights_outcome.items()},
    }
    print(f"\n  Ensemble: coef={ensemble_coef:.4f}, se={ensemble_se:.4f}")
    print(f"    Weights: {learner_results['Ensemble']['weights']}")
    
    # ── Best: method with lowest nuisance MSE ──
    best_outcome_method = min(methods_for_ensemble, key=lambda m: learner_nuisance_mse[m]["mse_outcome"])
    best_treatment_method = min(methods_for_ensemble, key=lambda m: learner_nuisance_mse[m]["mse_treatment"])
    
    ## Revision Round 1
    # FIX: When best_outcome != best_treatment, re-fit PLR with the
    # best-outcome learner for ml_l and the best-treatment learner for ml_m,
    # rather than copying the full result dict from best_outcome_method.
    # This ensures the "Best" row genuinely combines the two best nuisance
    # models, as claimed in the paper text.
    if best_outcome_method == best_treatment_method:
        # Same learner wins both — no need to re-fit
        learner_results["Best"] = {
            **{k: v for k, v in learner_results[best_outcome_method].items()},
            "best_outcome_method": best_outcome_method,
            "best_treatment_method": best_treatment_method,
            "selection_criterion": "lowest_nuisance_MSE",
        }
    else:
        # Different learners win — re-fit with mixed nuisance models
        print(f"\n  Re-fitting Best with ml_l={best_outcome_method}, ml_m={best_treatment_method}...")
        from sklearn.base import clone
        best_learners = build_learners(n_spec, len(controls))
        ml_l_best = clone(best_learners[best_outcome_method])
        ml_m_best = clone(best_learners[best_treatment_method])
        
        obj_best = dml.DoubleMLPLR(
            dml_data, ml_l=ml_l_best, ml_m=ml_m_best,
            n_folds=k_folds, n_rep=n_rep
        )
        obj_best.fit()
        
        best_coefs = obj_best.all_coef.flatten()
        best_ses = obj_best.all_se.flatten()
        best_median_coef = float(np.median(best_coefs))
        best_se_adj = float(np.median(np.sqrt(best_ses**2 + (best_coefs - best_median_coef)**2)))
        best_pval = float(2 * (1 - stats.norm.cdf(abs(best_median_coef / best_se_adj)))) if best_se_adj > 0 else 1.0
        
        # Nuisance R² for the mixed Best model
        best_preds_y = obj_best.predictions['ml_l'][:, :, 0].mean(axis=1)
        best_preds_d = obj_best.predictions['ml_m'][:, :, 0].mean(axis=1)
        best_r2_y = float(r2_score(df_spec[y_col], best_preds_y))
        best_r2_d = float(r2_score(df_spec[d_col], best_preds_d))
        
        learner_results["Best"] = {
            "coef": best_median_coef,
            "se": best_se_adj,
            "ci_lo": best_median_coef - 1.96 * best_se_adj,
            "ci_hi": best_median_coef + 1.96 * best_se_adj,
            "pval": best_pval,
            "per_rep_coefs": best_coefs.tolist(),
            "per_rep_ses": best_ses.tolist(),
            "nuisance": {
                "mse_outcome": float(mean_squared_error(df_spec[y_col], best_preds_y)),
                "mse_treatment": float(mean_squared_error(df_spec[d_col], best_preds_d)),
                "r2_outcome": best_r2_y,
                "r2_treatment": best_r2_d,
            },
            "best_outcome_method": best_outcome_method,
            "best_treatment_method": best_treatment_method,
            "selection_criterion": "lowest_nuisance_MSE",
        }
        
        # Store fitted Best object for HTE (primary spec only)
        if spec_idx == 0:
            learner_objects["Best"] = {
                "obj": obj_best, "preds_outcome": best_preds_y,
                "preds_treatment": best_preds_d, "df_spec": df_spec,
            }
        
        stars_b = "***" if best_pval < 0.01 else ("**" if best_pval < 0.05 else ("*" if best_pval < 0.10 else ""))
        print(f"  Best (re-fitted): coef={best_median_coef:.4f}, se={best_se_adj:.4f}, p={best_pval:.4f} {stars_b}")
        print(f"    Nuisance R²: outcome={best_r2_y:.3f}, treatment={best_r2_d:.3f}")
    
    print(f"  Best: {best_outcome_method} (outcome MSE), {best_treatment_method} (treatment MSE)")
    
    # ── OLS baseline from replication results ──
    ols_coef = None
    for r in rep['specs']:
        if d_col in r.get('spec', ''):
            ols_coef = r['replicated_coef']
            ols_se = r.get('replicated_se', None)
            break
    
    # Check sign change vs published
    published_coef = None
    for mr in spec.get('main_results', []):
        if d_col in mr.get('spec', '') and 'OLS' in mr.get('spec', ''):
            published_coef = mr.get('coef')
            break
    
    sign_change = False
    if published_coef is not None and published_coef != 0:
        sign_change = bool(np.sign(learner_results["Best"]["coef"]) != np.sign(published_coef))
    
    all_results.append({
        "outcome": s["outcome"],
        "treatment": s["treatment"],
        "n_obs": n_spec,
        "n_folds": k_folds,
        "n_rep": n_rep,
        "learners": learner_results,
        "ols_coef": ols_coef,
        "ols_se": ols_se,
        "preferred_learner": "Best",
        "preferred_coef": learner_results["Best"]["coef"],
        "preferred_ci_lo": learner_results["Best"]["ci_lo"],
        "preferred_ci_hi": learner_results["Best"]["ci_hi"],
        "sign_change_vs_published": sign_change,
    })

print(f"\n{'='*70}")
print(f"DML estimation complete: {len(all_results)} specifications")
print(f"{'='*70}")


Spec 1/3: growth ~ skill1_corr

  Fitting Lasso... 

Lasso selected 6/17 vars (outcome), 2/17 vars (treatment)
coef=0.0198, se=0.0133, p=0.1373 
    Nuisance R²: outcome=0.118, treatment=0.310

  Fitting DecisionTree... coef=0.0132, se=0.0112, p=0.2368 
    Nuisance R²: outcome=-0.030, treatment=0.272

  Fitting Boosting... 

coef=0.0197, se=0.0124, p=0.1108 
    Nuisance R²: outcome=0.154, treatment=0.362

  Fitting Forest... 

coef=0.0175, se=0.0105, p=0.0950 *
    Nuisance R²: outcome=0.074, treatment=0.273

  Fitting NeuralNet... 

coef=0.2668, se=0.0990, p=0.0071 ***
    Nuisance R²: outcome=-48.070, treatment=-0.180

  Ensemble: coef=0.0189, se=0.0123
    Weights: {'Lasso': 0.258, 'DecisionTree': 0.221, 'Boosting': 0.27, 'Forest': 0.246, 'NeuralNet': 0.005}
  Best: Boosting (outcome MSE), Boosting (treatment MSE)

Spec 2/3: growth ~ diffa

  Fitting Lasso... 

...

coef=0.1086, se=0.0447, p=0.0152 **
    Nuisance R²: outcome=-49.311, treatment=0.058

  Ensemble: coef=0.0092, se=0.0058
    Weights: {'Lasso': 0.249, 'DecisionTree': 0.224, 'Boosting': 0.274, 'Forest': 0.248, 'NeuralNet': 0.005}
  Best: Boosting (outcome MSE), Boosting (treatment MSE)

Spec 3/3: growth ~ diffg

  Fitting Lasso... 

Lasso selected 2/17 vars (outcome), 2/17 vars (treatment)
coef=0.0061, se=0.0057, p=0.2882 
    Nuisance R²: outcome=0.120, treatment=0.250

  Fitting DecisionTree... coef=0.0032, se=0.0049, p=0.5159 
    Nuisance R²: outcome=-0.041, treatment=0.230

  Fitting Boosting... 

coef=0.0064, se=0.0054, p=0.2342 
    Nuisance R²: outcome=0.178, treatment=0.294

  Fitting Forest... 

coef=0.0065, se=0.0050, p=0.1932 
    Nuisance R²: outcome=0.046, treatment=0.285

  Fitting NeuralNet... 

coef=0.1026, se=0.0488, p=0.0354 **
    Nuisance R²: outcome=-46.285, treatment=0.195

  Ensemble: coef=0.0061, se=0.0055
    Weights: {'Lasso': 0.259, 'DecisionTree': 0.219, 'Boosting': 0.278, 'Forest': 0.239, 'NeuralNet': 0.005}
  Best: Boosting (outcome MSE), Boosting (treatment MSE)

DML estimation complete: 3 specifications


In [4]:
# â”€â”€ Cell 4: Write dml_results.json â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

dml_results = {
    "model_type": model_type,
    "aggregation": "median_across_reps",
    "se_formula": "B&N adjusted: median(sqrt(SE_k^2 + (coef_k - median_coef)^2))",
    "specifications": []
}

for res in all_results:
    spec_out = {
        "outcome": res["outcome"],
        "treatment": res["treatment"],
        "n_obs": res["n_obs"],
        "n_folds": res["n_folds"],
        "n_rep": res["n_rep"],
        "learners": {},
        "ols_coef": res["ols_coef"],
        "ols_se": res["ols_se"],
        "preferred_learner": res["preferred_learner"],
        "preferred_coef": res["preferred_coef"],
        "preferred_ci_lo": res["preferred_ci_lo"],
        "preferred_ci_hi": res["preferred_ci_hi"],
        "sign_change_vs_published": res["sign_change_vs_published"],
    }
    for lname, lres in res["learners"].items():
        # Serialize without per_rep arrays for readability
        lr_out = {k: v for k, v in lres.items() if k not in ("per_rep_coefs", "per_rep_ses")}
        spec_out["learners"][lname] = lr_out
    dml_results["specifications"].append(spec_out)

DML_RESULTS.write_text(json.dumps(dml_results, indent=2))
print(f"Wrote {DML_RESULTS}")

# â”€â”€ Summary table â”€â”€
print(f"\n{'â”€'*80}")
print(f"{'Treatment':<15} {'Lasso':>8} {'Tree':>8} {'Boost':>8} {'Forest':>8} {'NNet':>8} {'Ens.':>8} {'Best':>8} {'OLS':>8}")
print(f"{'â”€'*80}")
for res in all_results:
    d_col = res['treatment']['variable']
    lr = res['learners']
    ols = res.get('ols_coef', None)
    ols_str = f"{ols:.3f}" if ols is not None else "N/A"
    print(f"{d_col:<15} {lr['Lasso']['coef']:>8.3f} {lr['DecisionTree']['coef']:>8.3f} "
          f"{lr['Boosting']['coef']:>8.3f} {lr['Forest']['coef']:>8.3f} "
          f"{lr['NeuralNet']['coef']:>8.3f} {lr['Ensemble']['coef']:>8.3f} "
          f"{lr['Best']['coef']:>8.3f} {ols_str:>8}")
print(f"{'â”€'*80}")
print("\nB&N reference (Panel A skill1_corr): Lasso=0.019, Boost=0.016, Forest=0.016, NNet=0.013, Ens=0.019, Best=0.016, OLS=0.035")
print("B&N reference (Panel B diffa):       Lasso=0.010, Boost=0.009, Forest=0.008, NNet=0.006, Ens=0.008, Best=0.008, OLS=0.016")
print("B&N reference (Panel C diffg):       Lasso=0.009, Boost=0.007, Forest=0.008, NNet=0.013, Ens=0.009, Best=0.008, OLS=0.020")

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\results\dml_results.json

â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
Treatment          Lasso     Tree    Boost   Forest     NNet     Ens.     Best      OLS
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
skill1_corr        0.020    0.013    0.020    0.018    0.267    0.019    0.020    0.035
diffa              0.008    0.006    0.011    0.010    0.109    0.009    0.011    0.016
diffg              0.006    0.003    0.006    0.006    0.103    0.006    0.006    0.019
â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â

In [5]:
# ── Cell 5: BLP heterogeneity test + GATE + CLAN (primary spec) ──────────────

# Use the primary specification (spec_idx=0: growth ~ skill1_corr)
primary_res = all_results[0]
primary_df = df_clean[[primary_res['outcome']['variable'], 
                        primary_res['treatment']['variable']] + controls].dropna().reset_index(drop=True)
y_col_p = primary_res['outcome']['variable']
d_col_p = primary_res['treatment']['variable']

## Revision Round 1
# FIX: Use best_outcome_method for ml_l and best_treatment_method for ml_m,
# consistent with the corrected Best learner logic in Cell 3.
best_outcome = primary_res['learners']['Best'].get('best_outcome_method', 'Forest')
best_treatment = primary_res['learners']['Best'].get('best_treatment_method', best_outcome)
print(f"HTE analysis using Best learner: ml_l={best_outcome}, ml_m={best_treatment}")

learners_dict = build_learners(len(primary_df), len(controls))
from sklearn.base import clone
ml_l_hte = clone(learners_dict[best_outcome])
ml_m_hte = clone(learners_dict[best_treatment])

dml_data_hte = dml.DoubleMLData(
    primary_df, y_col=y_col_p, d_cols=d_col_p, x_cols=controls
)

obj_hte = dml.DoubleMLPLR(
    dml_data_hte, ml_l=ml_l_hte, ml_m=ml_m_hte,
    n_folds=2, n_rep=1
)
obj_hte.fit()
print(f"HTE model fitted: coef={obj_hte.coef[0]:.4f}, se={obj_hte.se[0]:.4f}")

# ── 5a: BLP (Best Linear Predictor) heterogeneity test ──
# S(X) proxy: observation-level CATE approximation from cross-fitted residuals
preds_y = obj_hte.predictions['ml_l'][:, 0, 0]
preds_d = obj_hte.predictions['ml_m'][:, 0, 0]

Y_res = primary_df[y_col_p].values - preds_y
D_res = primary_df[d_col_p].values - preds_d

# Avoid division by zero for near-zero D_res
# S_proxy = (Y_i - Y_hat_i) / (D_i - D_hat_i)
safe_D_res = np.where(np.abs(D_res) < 1e-8, 1e-8, D_res)
S_proxy = Y_res / safe_D_res

# Winsorize extreme S_proxy values (small sample, residual ratios can be wild)
p5, p95 = np.percentile(S_proxy, [5, 95])
S_proxy = np.clip(S_proxy, p5, p95)

S_centered = S_proxy - np.mean(S_proxy)

# BLP regression: Y_res = beta1 * D_res + beta2 * D_res * (S - mean(S)) + eps
X_blp = np.column_stack([D_res, D_res * S_centered])
blp_model = sm.OLS(Y_res, X_blp).fit(cov_type='HC1')

blp_results = {
    "beta1_ate": float(blp_model.params[0]),
    "beta1_se": float(blp_model.bse[0]),
    "beta1_pval": float(blp_model.pvalues[0]),
    "beta2_het": float(blp_model.params[1]),
    "beta2_se": float(blp_model.bse[1]),
    "beta2_pval": float(blp_model.pvalues[1]),
    "heterogeneity_significant": bool(blp_model.pvalues[1] < 0.10),
}
print(f"\nBLP test:")
print(f"  beta1 (ATE) = {blp_results['beta1_ate']:.4f}, se={blp_results['beta1_se']:.4f}, p={blp_results['beta1_pval']:.4f}")
print(f"  beta2 (Het) = {blp_results['beta2_het']:.4f}, se={blp_results['beta2_se']:.4f}, p={blp_results['beta2_pval']:.4f}")
print(f"  Heterogeneity significant at 10%: {blp_results['heterogeneity_significant']}")

# ── 5b: GATE (Group Average Treatment Effects) — 5 quintiles ──
# Group by predicted CATE proxy quintiles
quintile_labels = ["Q1 (low)", "Q2", "Q3", "Q4", "Q5 (high)"]
try:
    quintiles = pd.qcut(S_proxy, q=5, labels=quintile_labels, duplicates='drop')
except ValueError:
    # If not enough unique values for 5 bins, try fewer
    quintiles = pd.qcut(S_proxy, q=3, labels=["Q1 (low)", "Q2-Q4", "Q5 (high)"], duplicates='drop')
    quintile_labels = quintiles.cat.categories.tolist()

# Build boolean group dummies for DoubleML GATE API
groups_df = pd.get_dummies(quintiles).astype(bool)
print(f"\nGATE groups: {list(groups_df.columns)}, sizes: {groups_df.sum().tolist()}")

gate = obj_hte.gate(groups=groups_df)
gate_ci = gate.confint(level=0.95, joint=True)

gate_groups = []
for i, label in enumerate(groups_df.columns):
    gate_groups.append({
        "label": str(label),
        "coef": float(gate_ci.iloc[i]['effect']),
        "ci_lo": float(gate_ci.iloc[i]['2.5 %']),
        "ci_hi": float(gate_ci.iloc[i]['97.5 %']),
    })

# Check for heterogeneity: any two CIs that don't overlap
heterogeneity_detected = False
for i in range(len(gate_groups)):
    for j in range(i+1, len(gate_groups)):
        if gate_groups[i]['ci_hi'] < gate_groups[j]['ci_lo'] or gate_groups[j]['ci_hi'] < gate_groups[i]['ci_lo']:
            heterogeneity_detected = True
            break

print(f"\nGATE results:")
for g in gate_groups:
    print(f"  {g['label']}: coef={g['coef']:.4f}, CI=[{g['ci_lo']:.4f}, {g['ci_hi']:.4f}]")
print(f"  Heterogeneity detected (non-overlapping CIs): {heterogeneity_detected}")

# ── 5c: CLAN (Classification Analysis) ──
top_q = S_proxy >= np.percentile(S_proxy, 80)
bot_q = S_proxy <= np.percentile(S_proxy, 20)

clan_results = []
for var in controls:
    vals_top = primary_df.loc[top_q, var].dropna()
    vals_bot = primary_df.loc[bot_q, var].dropna()
    mean_top = float(vals_top.mean()) if len(vals_top) > 0 else np.nan
    mean_bot = float(vals_bot.mean()) if len(vals_bot) > 0 else np.nan
    diff = mean_top - mean_bot
    
    if len(vals_top) > 1 and len(vals_bot) > 1:
        t_stat, p_val = stats.ttest_ind(vals_top, vals_bot, equal_var=False)
        p_val = float(p_val)
    else:
        p_val = 1.0
    
    clan_results.append({
        "variable": var,
        "mean_most_affected": round(mean_top, 4),
        "mean_least_affected": round(mean_bot, 4),
        "difference": round(diff, 4),
        "pval": round(p_val, 4),
        "significant_10pct": bool(p_val < 0.10),
    })

sig_clan = [c for c in clan_results if c['significant_10pct']]
print(f"\nCLAN: {len(sig_clan)}/{len(clan_results)} variables significantly different at 10%")
for c in sig_clan:
    print(f"  {c['variable']}: diff={c['difference']:.4f}, p={c['pval']:.4f}")

# CATE summary from S_proxy
cate_summary = {
    "mean": float(np.mean(S_proxy)),
    "sd": float(np.std(S_proxy)),
    "min": float(np.min(S_proxy)),
    "max": float(np.max(S_proxy)),
}

# ── Write hte_results.json ──
HTE_RESULTS = RESULTS_DIR / 'hte_results.json'
hte_results = {
    "blp": blp_results,
    "gate": {
        "method": "GATE",
        "grouping_variable": "predicted_CATE_quintile",
        "n_groups": len(gate_groups),
        "groups": gate_groups,
        "heterogeneity_detected": heterogeneity_detected,
    },
    "clan": clan_results,
    "cate_summary": cate_summary,
}
HTE_RESULTS.write_text(json.dumps(hte_results, indent=2))
print(f"\nWrote {HTE_RESULTS}")

HTE analysis using Best learner: ml_l=Boosting, ml_m=Boosting


HTE model fitted: coef=0.0170, se=0.0089

BLP test:
  beta1 (ATE) = 0.0063, se=0.0013, p=0.0000
  beta2 (Het) = 1.0987, se=0.0509, p=0.0000
  Heterogeneity significant at 10%: True

GATE groups: ['Q1 (low)', 'Q2', 'Q3', 'Q4', 'Q5 (high)'], sizes: [13, 12, 13, 12, 13]

GATE results:
  Q1 (low): coef=-0.2059, CI=[-0.3545, -0.0574]
  Q2: coef=-0.0218, CI=[-0.0494, 0.0058]
  Q3: coef=0.0140, CI=[-0.0052, 0.0333]
  Q4: coef=0.0760, CI=[0.0421, 0.1099]
  Q5 (high): coef=0.2349, CI=[0.0966, 0.3733]
  Heterogeneity detected (non-overlapping CIs): True

CLAN: 1/17 variables significantly different at 10%
  s_w_asia: diff=0.3077, p=0.0395

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\results\hte_results.json


In [6]:
# â”€â”€ Cell 6: Forest plot â€” coefficient comparison â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, ax = plt.subplots(figsize=(10, 6))

# Use primary specification
primary = all_results[0]
lr = primary['learners']

# Order: OLS, then DML methods, then Causal Forest (if available)
methods = ['Lasso', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet', 'Ensemble', 'Best']
labels = ['Lasso', 'Dec. Tree', 'Boosting', 'Random Forest', 'Neural Net', 'Ensemble', 'Best']

# Collect data
plot_data = []
for m, label in zip(methods, labels):
    r = lr[m]
    plot_data.append({
        'label': label, 'coef': r['coef'],
        'ci_lo': r['ci_lo'], 'ci_hi': r['ci_hi'],
        'color': '#2171b5'  # blue for DML
    })

# Add OLS
ols_coef = primary.get('ols_coef', None)
ols_se = primary.get('ols_se', None)
if ols_coef is not None:
    ols_ci_lo = ols_coef - 1.96 * (ols_se if ols_se else 0)
    ols_ci_hi = ols_coef + 1.96 * (ols_se if ols_se else 0)
    plot_data.append({
        'label': 'OLS', 'coef': ols_coef,
        'ci_lo': ols_ci_lo, 'ci_hi': ols_ci_hi,
        'color': '#636363'  # grey for OLS
    })

# Plot
y_positions = list(range(len(plot_data)))
for i, d in enumerate(plot_data):
    ax.errorbar(d['coef'], i, 
                xerr=[[d['coef'] - d['ci_lo']], [d['ci_hi'] - d['coef']]],
                fmt='o', color=d['color'], markersize=8, capsize=4, linewidth=2)

ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_yticks(y_positions)
ax.set_yticklabels([d['label'] for d in plot_data])
ax.set_xlabel('Coefficient estimate (95% CI)')
ax.set_title(f'DML Coefficient Comparison: {primary["treatment"]["variable"]} â†’ {primary["outcome"]["variable"]}\n'
             f'(N={primary["n_obs"]}, K={primary["n_folds"]}, n_rep={primary["n_rep"]})')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

fig.savefig(str(FIGURES_DIR / 'forest_plot.pdf'), dpi=300, bbox_inches='tight')
fig.savefig(str(FIGURES_DIR / 'forest_plot.png'), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"Wrote {FIGURES_DIR / 'forest_plot.pdf'}")
print(f"Wrote {FIGURES_DIR / 'forest_plot.png'}")

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\figures\forest_plot.pdf
Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\figures\forest_plot.png


In [7]:
# â”€â”€ Cell 7: B&N-style LaTeX comparison table (table_dml.tex) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def significance_stars(pval):
    if pval < 0.01: return "***"
    if pval < 0.05: return "**"
    if pval < 0.10: return "*"
    return ""

# Map treatment to panel labels (B&N Table 2 style)
panel_map = {
    "skill1_corr": "Panel A: Skill-tariff correlation (skill1\\_corr)",
    "diffa": "Panel B: Tariff differential, low cut-off (diffa)",
    "diffg": "Panel C: Tariff differential, high cut-off (diffg)",
}

methods_order = ['Lasso', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet', 'Ensemble', 'Best']
col_labels = ['Lasso', 'Tree', 'Boosting', 'Forest', 'NNet', 'Ensemble', 'Best', 'OLS']

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{DML estimates of tariff structure effects on growth (PLR, K=2, median aggregation)}")
lines.append(r"\small")
lines.append(r"\begin{tabular}{l" + "c" * 8 + "}")
lines.append(r"\toprule")
lines.append(" & " + " & ".join([f"({i+1})" for i in range(8)]) + r" \\")
lines.append(" & " + " & ".join(col_labels) + r" \\")
lines.append(r"\midrule")

for res in all_results:
    d_col = res['treatment']['variable']
    panel_label = panel_map.get(d_col, d_col)
    lines.append(r"\multicolumn{9}{l}{\textit{" + panel_label + r"}} \\")
    
    lr = res['learners']
    
    # Coefficient row
    coef_cells = []
    for m in methods_order:
        c = lr[m]['coef']
        stars = significance_stars(lr[m]['pval'])
        coef_cells.append(f"{c:.3f}{stars}")
    
    # OLS
    ols_c = res.get('ols_coef', None)
    if ols_c is not None:
        # Get OLS p-value from replication
        ols_pval = None
        for r in rep['specs']:
            if d_col in r.get('spec', ''):
                ols_pval = r.get('pvalue', None)
                break
        ols_stars = significance_stars(ols_pval) if ols_pval is not None else "***"
        coef_cells.append(f"{ols_c:.3f}{ols_stars}")
    else:
        coef_cells.append("--")
    
    lines.append(f"Treatment effect & " + " & ".join(coef_cells) + r" \\")
    
    # SE row
    se_cells = []
    for m in methods_order:
        se_cells.append(f"({lr[m]['se']:.3f})")
    ols_se_val = res.get('ols_se', None)
    se_cells.append(f"({ols_se_val:.3f})" if ols_se_val is not None else "--")
    lines.append(f" & " + " & ".join(se_cells) + r" \\")
    
    # N row
    lines.append(f"Observations & " + " & ".join([str(res['n_obs'])] * 8) + r" \\")
    lines.append(f"Controls & " + " & ".join([str(len(controls))] * 8) + r" \\")
    
    lines.append(r"\midrule")

# Remove last \midrule and replace with \bottomrule
lines[-1] = r"\bottomrule"
lines.append(r"\end{tabular}")
lines.append(r"\begin{tablenotes}")
lines.append(r"\small")
lines.append(r"\item Notes: DML2 with " + str(all_results[0]['n_folds']) + 
             r"-fold cross-fitting, " + str(all_results[0]['n_rep']) + 
             r" repetitions, median aggregation. SEs adjusted following Baiardi \& Naghi (2024).")
lines.append(r"\item * $p<0.10$, ** $p<0.05$, *** $p<0.01$.")
lines.append(r"\item Best learner selected by lowest out-of-sample nuisance MSE.")
lines.append(r"\end{tablenotes}")
lines.append(r"\end{table}")

table_dml_tex = "\n".join(lines)
(TABLES_DIR / 'table_dml.tex').write_text(table_dml_tex)
print(f"Wrote {TABLES_DIR / 'table_dml.tex'}")
print()
print(table_dml_tex)

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_dml.tex

\begin{table}[htbp]
\centering
\caption{DML estimates of tariff structure effects on growth (PLR, K=2, median aggregation)}
\small
\begin{tabular}{lcccccccc}
\toprule
 & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) \\
 & Lasso & Tree & Boosting & Forest & NNet & Ensemble & Best & OLS \\
\midrule
\multicolumn{9}{l}{\textit{Panel A: Skill-tariff correlation (skill1\_corr)}} \\
Treatment effect & 0.020 & 0.013 & 0.020 & 0.018* & 0.267*** & 0.019 & 0.020 & 0.035*** \\
 & (0.013) & (0.011) & (0.012) & (0.011) & (0.099) & (0.012) & (0.012) & (0.012) \\
Observations & 63 & 63 & 63 & 63 & 63 & 63 & 63 & 63 \\
Controls & 17 & 17 & 17 & 17 & 17 & 17 & 17 & 17 \\
\midrule
\multicolumn{9}{l}{\textit{Panel B: Tariff differential, low cut-off (diffa)}} \\
Treatment effect & 0.008 & 0.006 & 0.011* & 0.010* & 0.109** & 0.009 & 0.011* & 0.016*** \\
 & (0.006) & (0.005) & (0.006) & (0.005) 

In [8]:
# â”€â”€ Cell 8: GATE plot â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

fig, ax = plt.subplots(figsize=(8, 5))

gate_labels = [g['label'] for g in gate_groups]
gate_coefs = [g['coef'] for g in gate_groups]
gate_ci_los = [g['ci_lo'] for g in gate_groups]
gate_ci_his = [g['ci_hi'] for g in gate_groups]

y_pos = list(range(len(gate_labels)))
xerr_lo = [c - lo for c, lo in zip(gate_coefs, gate_ci_los)]
xerr_hi = [hi - c for c, hi in zip(gate_coefs, gate_ci_his)]

ax.errorbar(gate_coefs, y_pos, xerr=[xerr_lo, xerr_hi],
            fmt='o', color='#2171b5', markersize=8, capsize=5, linewidth=2)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(gate_labels)
ax.set_xlabel('Group Average Treatment Effect (95% joint CI)')
ax.set_title(f'GATE: {primary_res["treatment"]["variable"]} â†’ {primary_res["outcome"]["variable"]}\n'
             f'Groups defined by predicted CATE quintiles')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

fig.savefig(str(FIGURES_DIR / 'gate_plot.pdf'), dpi=300, bbox_inches='tight')
fig.savefig(str(FIGURES_DIR / 'gate_plot.png'), dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"Wrote {FIGURES_DIR / 'gate_plot.pdf'}")
print(f"Wrote {FIGURES_DIR / 'gate_plot.png'}")

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\figures\gate_plot.pdf
Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\figures\gate_plot.png


In [9]:
# â”€â”€ Cell 9: GATE LaTeX table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

gate_lines = []
gate_lines.append(r"\begin{table}[h]")
gate_lines.append(r"\centering")
gate_lines.append(r"\caption{Group Average Treatment Effects (GATE)}")
gate_lines.append(r"\begin{tabular}{lccc}")
gate_lines.append(r"\toprule")
gate_lines.append(r"Quintile & Coef. & 95\% CI (joint) \\")
gate_lines.append(r"\midrule")

for g in gate_groups:
    ci_str = f"[{g['ci_lo']:.3f}, {g['ci_hi']:.3f}]"
    gate_lines.append(f"{g['label']} & {g['coef']:.3f} & {ci_str} \\\\")

gate_lines.append(r"\bottomrule")
gate_lines.append(r"\end{tabular}")
gate_lines.append(r"\begin{tablenotes}")
gate_lines.append(r"\small")
gate_lines.append(r"\item Notes: Groups defined by quintiles of predicted CATE proxy.")
gate_lines.append(r"\item Jointly valid 95\% CIs via Gaussian multiplier bootstrap.")
het_str = "Yes" if heterogeneity_detected else "No"
gate_lines.append(r"\item Heterogeneity detected (non-overlapping CIs): " + het_str + ".")
gate_lines.append(r"\end{tablenotes}")
gate_lines.append(r"\end{table}")

gate_tex = "\n".join(gate_lines)
(TABLES_DIR / 'table_gate.tex').write_text(gate_tex)
print(f"Wrote {TABLES_DIR / 'table_gate.tex'}")
print()
print(gate_tex)

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_gate.tex

\begin{table}[h]
\centering
\caption{Group Average Treatment Effects (GATE)}
\begin{tabular}{lccc}
\toprule
Quintile & Coef. & 95\% CI (joint) \\
\midrule
Q1 (low) & -0.206 & [-0.355, -0.057] \\
Q2 & -0.022 & [-0.049, 0.006] \\
Q3 & 0.014 & [-0.005, 0.033] \\
Q4 & 0.076 & [0.042, 0.110] \\
Q5 (high) & 0.235 & [0.097, 0.373] \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\small
\item Notes: Groups defined by quintiles of predicted CATE proxy.
\item Jointly valid 95\% CIs via Gaussian multiplier bootstrap.
\item Heterogeneity detected (non-overlapping CIs): Yes.
\end{tablenotes}
\end{table}


In [10]:
# â”€â”€ Cell 10: CLAN LaTeX table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def pval_stars(pval):
    if pval < 0.01: return "***"
    if pval < 0.05: return "**"
    if pval < 0.10: return "*"
    return ""

clan_lines = []
clan_lines.append(r"\begin{table}[h]")
clan_lines.append(r"\centering")
clan_lines.append(r"\caption{Classification Analysis (CLAN): Characteristics of Most vs.\ Least Affected}")
clan_lines.append(r"\small")
clan_lines.append(r"\begin{tabular}{lcccc}")
clan_lines.append(r"\toprule")
clan_lines.append(r"Variable & Most Affected & Least Affected & Difference & p-value \\")
clan_lines.append(r"\midrule")

for c in clan_results:
    var_escaped = c['variable'].replace('_', r'\_')
    stars = pval_stars(c['pval'])
    clan_lines.append(
        f"{var_escaped} & {c['mean_most_affected']:.3f} & {c['mean_least_affected']:.3f} & "
        f"{c['difference']:.3f} & {c['pval']:.3f}{stars} \\\\"
    )

clan_lines.append(r"\bottomrule")
clan_lines.append(r"\end{tabular}")
clan_lines.append(r"\begin{tablenotes}")
clan_lines.append(r"\small")
clan_lines.append(r"\item Notes: Most affected = top quintile of predicted CATE; least affected = bottom quintile.")
clan_lines.append(r"\item Welch two-sample t-test. * $p<0.10$, ** $p<0.05$, *** $p<0.01$.")
clan_lines.append(r"\end{tablenotes}")
clan_lines.append(r"\end{table}")

clan_tex = "\n".join(clan_lines)
(TABLES_DIR / 'table_clan.tex').write_text(clan_tex)
print(f"Wrote {TABLES_DIR / 'table_clan.tex'}")
print()
print(clan_tex)

Wrote C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_clan.tex

\begin{table}[h]
\centering
\caption{Classification Analysis (CLAN): Characteristics of Most vs.\ Least Affected}
\small
\begin{tabular}{lcccc}
\toprule
Variable & Most Affected & Least Affected & Difference & p-value \\
\midrule
avg\_tar & -2.092 & -1.337 & -0.754 & 0.311 \\
init & 8.144 & 7.812 & 0.332 & 0.415 \\
inv & 2.813 & 2.317 & 0.496 & 0.115 \\
human\_cap & -2.452 & -2.732 & 0.280 & 0.547 \\
w\_africa & 0.000 & 0.154 & -0.154 & 0.165 \\
e\_africa & 0.000 & 0.154 & -0.154 & 0.165 \\
s\_c\_africa & 0.077 & 0.000 & 0.077 & 0.337 \\
n\_afr\_me & 0.154 & 0.154 & 0.000 & 1.000 \\
e\_europe & 0.077 & 0.077 & 0.000 & 1.000 \\
lat\_america & 0.077 & 0.154 & -0.077 & 0.558 \\
e\_asia & 0.077 & 0.154 & -0.077 & 0.558 \\
s\_e\_asia & 0.154 & 0.077 & 0.077 & 0.558 \\
s\_w\_asia & 0.308 & 0.000 & 0.308 & 0.040** \\
dum80\_83 & 0.615 & 0.769 & -0.154 & 0.416 \\
dum85\_87 & 0.308

In [11]:
# â”€â”€ Cell 11: Causal Forest (skip â€” econml not installed) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

if ECONML_AVAILABLE:
    print("econml is available â€” running CausalForestDML...")
    # Would run CausalForestDML here (see skills/stage_04.md section 8)
else:
    print("econml not installed â€” skipping Causal Forest estimation.")
    print("To enable: pip install econml")
    print("Causal forest outputs (cate_histogram.pdf, feature_importance.pdf, causal_forest_results.json) will not be generated.")

# â”€â”€ Final summary â”€â”€
print("\n" + "=" * 70)
print("STAGE 04 â€” DML EXTENSION COMPLETE")
print("=" * 70)
print(f"\nOutputs written:")
print(f"  {DML_RESULTS}")
print(f"  {RESULTS_DIR / 'hte_results.json'}")
print(f"  {TABLES_DIR / 'table_dml.tex'}")
print(f"  {TABLES_DIR / 'table_gate.tex'}")
print(f"  {TABLES_DIR / 'table_clan.tex'}")
print(f"  {FIGURES_DIR / 'forest_plot.pdf'}")
print(f"  {FIGURES_DIR / 'forest_plot.png'}")
print(f"  {FIGURES_DIR / 'gate_plot.pdf'}")
print(f"  {FIGURES_DIR / 'gate_plot.png'}")
print(f"\nKey findings:")
for res in all_results:
    d = res['treatment']['variable']
    best = res['learners']['Best']
    ols = res.get('ols_coef', None)
    ratio = best['coef'] / ols if ols and ols != 0 else None
    ratio_str = f" ({ratio:.0%} of OLS)" if ratio is not None else ""
    print(f"  {d}: Best={best['coef']:.4f} (se={best['se']:.4f}), OLS={ols:.4f}{ratio_str}")
print(f"\nSign change vs published: {any(r['sign_change_vs_published'] for r in all_results)}")

econml not installed â€” skipping Causal Forest estimation.
To enable: pip install econml
Causal forest outputs (cate_histogram.pdf, feature_importance.pdf, causal_forest_results.json) will not be generated.

STAGE 04 â€” DML EXTENSION COMPLETE

Outputs written:
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\results\dml_results.json
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\results\hte_results.json
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_dml.tex
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_gate.tex
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\tables\table_clan.tex
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\paper\figures\forest_plot.pdf
  C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefle